# Clouds Decoded — Getting Started

This notebook walks you through **clouds-decoded**, an end-to-end toolkit for
retrieving cloud physical properties from Sentinel-2 satellite imagery.

By the end of this notebook you will have:
1. Installed clouds-decoded and downloaded model weights
2. Queried and downloaded Sentinel-2 scenes from the Copernicus Data Space
3. Run the full processing pipeline on individual scenes
4. Set up a **project** for batch processing of multiple scenes

### What does clouds-decoded do?

The Copernicus Sentinel-2 satellites are operated by ESA. Each one
carries a push-broom multispectral imager that captures 13 spectral bands from
visible to shortwave infrared, at resolutions of 10–60 m. The raw product
(Level-1C) contains top-of-atmosphere reflectance in `.SAFE` directories.

**clouds-decoded** takes these `.SAFE` directories and produces georeferenced
outputs:

| Step | What it does | Output |
|------|-------------|--------|
| **Cloud Mask** | 4-class segmentation (clear / thick cloud / thin cloud / shadow) | `cloud_mask.tif` |
| **Cloud Height** | Estimates cloud-top height in metres | `cloud_height.tif` |
| **Albedo** | Estimates surface reflectance beneath clouds (only done for input into cloud properties module) | `albedo.tif` |
| **Refocus** | Corrects parallax misalignment between bands | *(in-memory)* |
| **Cloud Properties** | Retrieves optical thickness, droplet size, ice fraction | `cloud_properties.tif` |

Let's get started!

## 1. Installation and Setup

Install clouds-decoded and download the model weights and a sample scene.
If you've already done this, the download commands will skip existing files.

In [ ]:
# Install clouds-decoded (skip if already installed)
%pip install -e .. -q

In [ ]:
# Download model weights and the sample Sentinel-2 scene.
# Each asset is ~100-700 MB. Already-downloaded files are skipped.

# Attention: This will download ~2 GB of data, so make sure you have enough disk space and a good internet connection. It will be placed, by default, in your user home directory under `~/.local/share/clouds-decoded/`.
!clouds-decoded download all --yes

In [ ]:
# Locate the sample scene
from clouds_decoded.assets import require_asset

scene_path = str(require_asset("sample_scene"))
print(f"Sample scene: {scene_path}")

## 2. Load and Visualise the Scene

Load the `.SAFE` directory and take a look at the true-colour and
ice-composite views.

In [ ]:
from clouds_decoded.data import Sentinel2Scene

scene = Sentinel2Scene()
scene.read(scene_path)

# Parse identifiers from the .SAFE directory name
safe_name = scene.scene_directory.name  # e.g. S2B_MSIL1C_20250104T185019_...SAFE
parts = safe_name.replace(".SAFE", "").split("_")
print(f"Scene:     {safe_name}")
print(f"Satellite: {parts[0]}")
print(f"Time:      {scene.sensing_time}")
print(f"CRS:       {scene.crs}")
print(f"Bands:     {list(scene.bands.keys())}")

In [ ]:
import matplotlib.pyplot as plt
from clouds_decoded.visualisation.layers import layer_from_rgb
from clouds_decoded.visualisation.static import render_to_axes

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

rgb_layer = layer_from_rgb(scene, "true_color", display_resolution_m=60)
ice_layer = layer_from_rgb(scene, "ice", display_resolution_m=60)

render_to_axes(rgb_layer, ax1)
render_to_axes(ice_layer, ax2)
fig.tight_layout()
plt.show()

## 3. Run the Processing Pipeline

The quickest way to process a scene is the `full-workflow` CLI command:

```bash
clouds-decoded full-workflow /path/to/scene.SAFE --output-dir output/
```

This runs all five steps in sequence. But here we'll go step by step
so you can see each stage in detail.

In [ ]:
!clouds-decoded full-workflow {scene_path} --output-dir outputs/full_workflow

### Visualise the results

The `Visualiser` class loads all outputs from a scene directory and
auto-detects how to render them.  The `overview()` method plots
everything in a grid.

In [ ]:
from clouds_decoded.visualisation.visualiser import Visualiser

vis = Visualiser.from_directory("outputs/full_workflow", scene_path=scene_path)
print(f"Loaded layers: {vis.layer_names}")

fig = vis.overview(ncols=3, figsize=(20, 16))
plt.show()

### Composites

You can overlay any property on the RGB base. Here we show optical
thickness over the ice composite.

In [ ]:
fig = vis.composite("Ice Composite", "Properties: tau", alpha=0.5, figsize=(12, 10))
plt.show()

## 4. Running Individual Steps (Python API)

For more control you can run each processing step individually.
This is useful when you want to inspect intermediate results or
customise configurations.

In [ ]:
from pathlib import Path

# Create output directory for step-by-step results
step_out = Path("outputs/step_by_step")
step_out.mkdir(exist_ok=True, parents=True)

### Step 1: Cloud Mask

Segments each pixel into clear, thick cloud, thin cloud, or shadow
using a SegFormer-B2 model trained on Sentinel-2.

In [ ]:
from clouds_decoded.modules.cloud_mask.config import CloudMaskConfig
from clouds_decoded.modules.cloud_mask.processor import CloudMaskProcessor

mask_config = CloudMaskConfig(
    method="senseiv2",  # This is the default, but left here for clarity. If you do not have a GPU, you can switch to "threshold" for a simple value thresholding method, which is much faster but less accurate.
    batch_size=4,        # Adjust based on your system's memory capacity, 4 is a good starting point for most modern GPUs with 8 GB VRAM or more.
    )
mask_processor = CloudMaskProcessor(mask_config)

mask_result = mask_processor.process(scene)
mask_result.write(str(step_out / "cloud_mask.tif"))
print(f"Cloud mask shape: {mask_result.data.shape}")
print(f"Classes: {mask_result.metadata.classes}")

In [ ]:
from clouds_decoded.visualisation.layers import layer_from_cloud_mask
from clouds_decoded.visualisation.static import plot_layer

fig = plot_layer(layer_from_cloud_mask(mask_result), figsize=(8, 8))
plt.show()

### Step 2: Cloud Height

Estimates cloud-top height in metres using a deep learning emulator
(ResUNet) that approximates the stereo-parallax algorithm.

In [ ]:
from clouds_decoded.modules.cloud_height_emulator.config import CloudHeightEmulatorConfig
from clouds_decoded.modules.cloud_height_emulator.processor import CloudHeightEmulatorProcessor

height_config = CloudHeightEmulatorConfig()
height_processor = CloudHeightEmulatorProcessor(height_config)

# The emulator uses the cloud mask to focus on cloudy regions
binary_mask = mask_result.to_binary(positive_classes=[1, 2], threshold=0.2)
height_result = height_processor.process(scene, cloud_mask=binary_mask)
height_result.write(str(step_out / "cloud_height.tif"))
print(f"Height map shape: {height_result.data.shape}")

In [ ]:
from clouds_decoded.visualisation.layers import layer_from_cloud_height

fig = plot_layer(layer_from_cloud_height(height_result), figsize=(8, 8))
plt.show()

### Steps 3–5: Albedo, Refocus, and Cloud Properties

The remaining steps run in sequence: estimate surface albedo,
correct parallax misalignment (refocus), then invert reflectance
to retrieve cloud optical and microphysical properties.

In [ ]:
from clouds_decoded.modules.albedo_estimator.config import AlbedoEstimatorConfig
from clouds_decoded.modules.albedo_estimator.processor import AlbedoEstimator

albedo_config = AlbedoEstimatorConfig(method="idw")
albedo_processor = AlbedoEstimator(albedo_config)

# Use a broader mask (include shadow) for albedo clear-sky sampling
albedo_mask = mask_result.to_binary(positive_classes=[1, 2, 3])
albedo_result = albedo_processor.process(scene, cloud_mask=albedo_mask)
albedo_result.write(str(step_out / "albedo.tif"))
print(f"Albedo shape: {albedo_result.data.shape}, bands: {albedo_result.metadata.band_names}")

In [ ]:
from clouds_decoded.modules.refocus.config import RefocusConfig
from clouds_decoded.modules.refocus.processor import RefocusProcessor

refocus_config = RefocusConfig(
    output_resolution=None # Keep original resolution of each band, rather than resampling to a common resolution. Usually, this would just be set to 60 m/pixel for the cloud properties processor, but for demonstration we keep the original resolutions here for plotting.
) 
refocus_processor = RefocusProcessor(refocus_config)

# Refocus produces a corrected scene in memory (no output file)
refocused_scene = refocus_processor.process(scene, height_data=height_result)
print(f"Refocused: {refocused_scene.is_refocused}")

### Refocus A/B Comparison

The push-broom sensor acquires each band at a slightly different time,
so moving clouds appear shifted between bands. The refocus step warps
each band back to a common reference, removing this parallax.

Below we crop a small patch and compare the true-colour RGB before
and after refocusing. Adjust `crop_window` to find an area with
visible cloud — the parallax shift is easiest to see at sharp cloud
edges.

In [ ]:
import numpy as np
from clouds_decoded.visualisation.layers import rgb_config_from_data, _apply_rgb_config
from clouds_decoded.visualisation.static import _inset_title, render_to_axes

# ── Crop window (row_start, row_end, col_start, col_end) in B02 10 m pixels ──
# Adjust these to centre on an area with cloud edges.
r0, r1, c0, c1 = 5500, 7000, 0, 1500

# Extract RGB bands for the crop from both scenes
def _crop_rgb(sc, r0, r1, c0, c1):
    r = sc.get_band("B04", reflectance=True)[r0:r1, c0:c1]
    g = sc.get_band("B03", reflectance=True)[r0:r1, c0:c1]
    b = sc.get_band("B02", reflectance=True)[r0:r1, c0:c1]
    return np.stack([r, g, b], axis=-1).astype(np.float32)

rgb_before = _crop_rgb(scene, r0, r1, c0, c1)
rgb_after  = _crop_rgb(refocused_scene, r0, r1, c0, c1)

# Shared contrast stretch so both panels are comparable
cfg = rgb_config_from_data(rgb_before)
rgb_before = _apply_rgb_config(rgb_before, cfg)
rgb_after  = _apply_rgb_config(rgb_after, cfg)

# A/B plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))
ax1.imshow(rgb_before); ax1.set_xticks([]); ax1.set_yticks([])
ax2.imshow(rgb_after);  ax2.set_xticks([]); ax2.set_yticks([])
_inset_title(ax1, "Before refocus")
_inset_title(ax2, "After refocus")
fig.tight_layout(pad=0.5)
plt.show()

In [ ]:
from clouds_decoded.modules.refl2prop.config import ShadingRefl2PropConfig
from clouds_decoded.modules.refl2prop.processor import CloudPropertyInverter

props_config = ShadingRefl2PropConfig()
props_processor = CloudPropertyInverter(props_config)

# Cloud properties use the refocused scene, height map, and albedo
props_result = props_processor.process(
    refocused_scene,
    height_data=height_result,
    albedo_data=albedo_result,
)
props_result.write(str(step_out / "properties.tif"))
print(f"Properties shape: {props_result.data.shape}")
print(f"Bands: {props_result.metadata.band_names}")

In [ ]:
# Visualise the step-by-step results
vis_steps = Visualiser.from_directory(str(step_out), scene_path=scene_path)
fig = vis_steps.overview(ncols=3, figsize=(20, 16))
plt.show()

## 5. Batch Processing with Projects

For processing multiple scenes, use the **project** system. It handles
config management, resumability, and statistics collection.

### Initialise a project

In [ ]:
# Create a new project
!clouds-decoded project init outputs/my_project --name "Demo Project"

### Run the project

Pass one or more `.SAFE` paths to `project run`. The project system
automatically stages them, runs the pipeline, and computes statistics.
If a scene has already been processed, it will be skipped.

In [ ]:
# Process the sample scene within the project
!clouds-decoded project run outputs/my_project {scene_path}

In [ ]:
# Check project status
!clouds-decoded project status outputs/my_project

### Browse project results

The `ProjectVisualiser` gives you an interactive browser-based viewer
with scene navigation.  In a notebook, use `.panel()` for inline display,
or `.serve()` to launch a standalone server.

In [ ]:
from clouds_decoded.visualisation.project_visualiser import ProjectVisualiser

pv = ProjectVisualiser("outputs/my_project")
print(f"Scenes: {pv.scene_ids}")

# Static overview of the first scene
fig = pv.overview(pv.scene_ids[0], ncols=3, figsize=(20, 16))
plt.show()

In [ ]:
# Interactive panel viewer (requires panel and bokeh)
# Uncomment to launch inline:
!pip install jupyter_bokeh
import panel as pn; pn.extension()
pv.panel()

# Or launch as a standalone server:
# pv.serve(port=5006, show=True)

## Next Steps

- **Custom configs**: Edit the YAML files in `outputs/my_project/configs/` to
  change processing parameters, then re-run with `project run`.
- **More scenes**: Download scenes from the
  [Copernicus Data Space](https://dataspace.copernicus.eu/) and pass them
  to `project run`.
- **Statistics**: View per-scene statistics with
  `clouds-decoded project stats outputs/my_project`.
- **CLI reference**: Run `clouds-decoded --help` for all available commands.